# 02. Matching monotowns to OKTMO codes

**Goal:** find the official code of each monotown so it can be joined with statistics.

**What is OKTMO?** OKTMO (Russian Classification of Territories of Municipal Formations) is the official ID number of every municipality in Russia. Names of the same place vary between documents, but the code stays the same, so all joins in this project use codes.

The code is hierarchical, from large to small. Example, Slantsy:
- `41` Leningrad Oblast
- `41642000` Slantsy municipal district
- `41642101` Slantsy urban settlement (the town itself)

In [3]:
import pandas as pd
data = "../data/"

mono = pd.read_csv(data + "monotowns_2026.csv", encoding="utf-8-sig")
mono["category"] = pd.cut(mono["number"], bins=[0, 52, 201, 264], labels=[1, 2, 3])

ref = pd.read_csv(data + "processed/population.csv", dtype={"oktmo": str, "oktmo_stable": str})
ref = ref[ref["year"] == 2025]
print(ref.shape)

(13747, 8)


In [5]:
examples = "Райчихинск|Онежский|Воркута|Тольятти|Кондопож|Сланцев|Жешарт|Нерюнгри"
ref[ref["municipality"].str.contains(examples, na=False)][
    ["region_name", "mun_district", "municipality", "mun_type", "oktmo"]]

,region_name,mun_district,municipality,mun_type,oktmo
18544,Амурская область,город Райчихинск,город Райчихинск,"Городской округ, городской округ с внутригород...",10720000
18559,Архангельская область,Онежский муниципальный округ,Онежский муниципальный округ,Муниципальный округ,11546000
22750,Ленинградская область,Сланцевский муниципальный район,Сланцевский муниципальный район,Муниципальный район,41642000
22751,Ленинградская область,Сланцевский муниципальный район,Сланцевское,Городское поселение,41642101
28132,Республика Карелия,Кондопожский муниципальный район,Кондопожский муниципальный район,Муниципальный район,86615000
28133,Республика Карелия,Кондопожский муниципальный район,Кондопожское,Городское поселение,86615101
28218,Республика Коми,Муниципальный округ Воркута,Муниципальный округ Воркута,Муниципальный округ,87510000
28355,Республика Коми,Усть-Вымский муниципальный район,Жешарт,Городское поселение,87644155
30688,Республика Саха (Якутия),Нерюнгринский муниципальный район,Нерюнгринский муниципальный район,Муниципальный район,98660000
30689,Республика Саха (Якутия),Нерюнгринский муниципальный район,Город Нерюнгри,Городское поселение,98660101


In [7]:
missing = sorted(set(mono["region"]) - set(ref["region_name"]))
print(missing)

['Донецкая Народная Республика', 'Запорожская область', 'Кемеровская область - Кузбасс', 'Луганская Народная Республика', 'Республика Бурятия', 'Республика Дагестан', 'Республика Крым']


In [9]:
import re

STOP = {"муниципальное", "образование", "муниципальный", "муниципального", "округ",
        "район", "района", "городское", "городской", "городского", "поселение",
        "город", "поселок", "рабочий", "типа", "закрытое", "административно-территориальное",
        "подведомственной", "территорией", "административным", "центром", "области",
        "края", "республики", "народной", "автономной", "кузбасса", "поссовет", "улус"}

def clean_words(text):
    text = re.sub(r"[^а-я\- ]", " ", str(text).lower().replace("ё", "е"))
    words = [w.strip("-") for w in text.split()]
    return [w for w in words if w not in STOP and len(w) > 2]

def stem(word):
    if "-" in word:                      # Усть-Илимск -> "усть-илим"
        a, b = word.split("-", 1)
        return a + "-" + b[:4]
    return word[:5]                      # Сланцевское -> "сланц"

def mono_type(name):
    n = name.lower()
    if "поселени" in n or "района" in n or "улус" in n:
        return "Городское поселение"
    if "муниципальный округ" in n or "муниципальный район" in n:
        return "Муниципальный округ"
    return "Городской округ, городской округ с внутригородским делением"

UPPER = ["Муниципальный округ", "Муниципальный район",
         "Городской округ, городской округ с внутригородским делением"]

#root of each Rosstat name = root of its longest distinctive word
ref["stem"] = ref["municipality"].apply(
    lambda n: stem(max(clean_words(n), key=len)) if clean_words(n) else None)

results = []
for _, m in mono.iterrows():
    region_stems = {stem(w) for w in clean_words(m["region"])}
    words = [w for w in clean_words(m["municipality"]) if stem(w) not in region_stems]
    text = " ".join(words)
    target = mono_type(m["municipality"])

    cands = ref[(ref["region_name"] == m["region"]) & ref["stem"].notna()].copy()
    allowed = ["Городское поселение"] if target == "Городское поселение" else UPPER
    cands = cands[cands["mun_type"].isin(allowed)]
    cands["pos"] = cands["stem"].apply(lambda s: text.find(s))
    cands = cands[cands["pos"] >= 0]
    cands["type_ok"] = cands["mun_type"] == target
    cands = cands.sort_values(["type_ok", "pos"], ascending=[False, True])

    best = cands.iloc[0] if len(cands) else None
    results.append({
        "number": m["number"],
        "mono_name": m["municipality"],
        "region": m["region"],
        "n_candidates": len(cands),
        "ref_name": best["municipality"] if best is not None else None,
        "ref_type": best["mun_type"] if best is not None else None,
        "oktmo": best["oktmo"] if best is not None else None,
    })

match = pd.DataFrame(results)
print("matched:", match["oktmo"].notna().sum(), "of", len(match))
print("several candidates:", (match["n_candidates"] > 1).sum())
match[match["oktmo"].isna()][["number", "mono_name", "region"]]

matched: 216 of 264
several candidates: 61


,number,mono_name,region
3,4,Муниципальное образование городской округ Енак...,Донецкая Народная Республика
4,5,Городской округ Мариуполь Донецкой Народной Ре...,Донецкая Народная Республика
5,6,Городской округ Торез Донецкой Народной Респуб...,Донецкая Народная Республика
6,7,Шахтерский муниципальный округ Донецкой Народн...,Донецкая Народная Республика
9,10,Городской округ Энергодар Запорожской области,Запорожская область
10,11,Анжеро-Судженский городской округ Кемеровской ...,Кемеровская область - Кузбасс
11,12,Березовский городской округ Кемеровской област...,Кемеровская область - Кузбасс
12,13,Калтанский городской округ Кемеровской области...,Кемеровская область - Кузбасс
13,14,Киселевский городской округ Кемеровской област...,Кемеровская область - Кузбасс
14,15,Междуреченский муниципальный округ Кемеровской...,Кемеровская область - Кузбасс
